In [17]:
import os, time
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()                       # reads .env into environment variables
client = OpenAI()                   # picks up OPENAI_API_KEY automatically
MODEL = "gpt-5.6-luna"              # replace with the exact ID from your dashboard

In [18]:
## prompt
PROMPT = """You classify US consumer complaints about financial products.
Choose exactly one category from this list:

- bank_account: checking or savings accounts: opening, closing, locked or frozen accounts, deposits, overdraft fees, unauthorized debits from the account
- money_transfer: sending money through apps or services (Zelle, PayPal, Venmo, Cash App, Western Union, wire transfers), virtual currency, money orders: payments not received, sent to the wrong person, scams paid this way, transfer fees
- credit_card: credit cards and prepaid cards, including charges, fees, rewards, card disputes
- credit_reporting: errors on credit reports, credit bureaus, credit repair services, identity theft on a report
- debt_collection: a collector or company trying to collect a debt the consumer owes or disputes
- mortgage: home loans, refinancing, escrow, foreclosure, loan servicing
- student_loan: federal or private student loans and their servicers

If the complaint is mainly about a transfer service or a payment sent to someone, choose money_transfer. If it is mainly about the account itself, choose bank_account.
Reply with only the category name, exactly as written above. No other text.

Complaint:
{complaint}
"""

In [19]:
## Classify function
LABELS = ["bank_account", "money_transfer", "credit_card", "credit_reporting",
          "debt_collection", "mortgage", "student_loan"]

def classify(text):
    response = client.responses.create(
        model=MODEL,
        input=PROMPT.format(complaint=text),
    )
    answer = response.output_text.strip().lower()
    if answer not in LABELS:
        return "invalid", answer, response.usage
    return answer, answer, response.usage

In [20]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset("liri-uzh/cfpb-complaints-mini")
train = ds["train"].to_pandas()
test = ds["test"].to_pandas()
train.shape, test.shape
train.head(2)

,text,label,product_raw
0,I use Sallie Mae for my student loans and if I...,student_loan,Student loan
1,i received a statement from aidvanatge today s...,student_loan,Student loan


In [21]:
classify(train["text"][0])

('student_loan',
 'student_loan',
 ResponseUsage(input_tokens=392, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=7, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=399))

In [40]:
def run_batch(df):
    rows = []
    for text, true_label in zip(df["text"], df["label"]):
        start = time.perf_counter()
        pred, raw, usage = classify(text)  # Sends each complaint to the LLM, one at a time
        latency_ms = (time.perf_counter() - start) * 1000
        rows.append({
            "true": true_label,
            "pred": pred, 
            "raw": raw, 
            "input_tokens": usage.input_tokens, 
            "output_tokens": usage.output_tokens,
            "time": latency_ms
        })
    return pd.DataFrame(rows)

In [41]:
dev = train.groupby("label").head(3)
results = run_batch(dev)
results

,true,pred,raw,input_tokens,output_tokens,time
0,student_loan,student_loan,student_loan,392,7,866.835000
1,student_loan,student_loan,student_loan,274,7,874.539125
2,debt_collection,debt_collection,debt_collection,337,7,1608.668500
3,debt_collection,credit_reporting,credit_reporting,418,6,953.375625
4,credit_reporting,credit_reporting,credit_reporting,335,6,1031.738083
5,credit_reporting,credit_reporting,credit_reporting,1413,6,795.882167
6,debt_collection,debt_collection,debt_collection,301,7,655.822584
7,credit_card,credit_card,credit_card,615,6,767.375042
8,student_loan,student_loan,student_loan,592,7,939.853125
9,money_transfer,money_transfer,money_transfer,388,55,2825.046583


In [42]:
test_results = run_batch(test)
test_results.to_csv("../data/llm_test_results.csv", index=False)

In [45]:
from sklearn.metrics import f1_score, classification_report

print(f1_score(test_results["true"], test_results["pred"], average="macro"))
print(classification_report(test_results["true"], test_results["pred"]))
print("invalid:", (test_results["pred"] == "invalid").sum())
print("median ms:", test_results["time"].median())
print("p95 ms:", test_results["time"].quantile(0.95))

0.7939687925161963
                  precision    recall  f1-score   support

    bank_account       0.72      0.83      0.77       240
     credit_card       0.87      0.68      0.77       219
credit_reporting       0.54      0.98      0.70       216
 debt_collection       0.90      0.47      0.61       240
  money_transfer       0.85      0.78      0.81       240
        mortgage       0.98      0.91      0.94       240
    student_loan       0.98      0.93      0.96       240

        accuracy                           0.80      1635
       macro avg       0.83      0.80      0.79      1635
    weighted avg       0.84      0.80      0.80      1635

invalid: 0
median ms: 946.2273330136668
p95 ms: 1935.8154920948432


## Comparison Table

| Approach | Macro F1 | Debt Coll Recall | Mortgage Recall | P95 Latency | Deterministic |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **A1. TF-IDF + LR** | 0.831 | 0.80 | 0.90 | 0.4 ms | yes |
| **A2. bge-small + LR** | 0.825 | 0.80 | 0.92 | 191 ms | yes |
| **B. Zero-shot LLM** | 0.794 | 0.47 | 0.91 | 1,936 ms | no |